# jopt_driver on a PYNQ board (ZCU104 or PYNQ-Z2)

The C++ driver (`software/driver/`) classifying a capture with the kernel in this folder
(`justounetsimple_opt` on the ZCU104, `justounetsimple_opt_zynq7020` on the PYNQ-Z2), from
`jopt_driver_BOARD.tar.gz`. The notebook only loads the bitstream (PYNQ also sets the FPGA
clock from the `.hwh`) and lends the driver DDR; reading the cube, running the kernel and
checking the answer is the driver's own work, as it would be without PYNQ.

Needs the image folder `aegean_unet` (from `tools/justounetsimple_image.py prepare`, the
same one the kernel's own notebook uses) next to this folder or in it. Shut down other
notebooks first: they share PYNQ's memory for buffers. Then **Run All**.

In [ ]:
import os, subprocess, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

# The image folder: next to this folder (where the justounetsimple_opt notebook unpacked
# it), or here; unpacked from its zip if only that is there.
for place in (Path('..'), Path('.')):
    if not (place / 'aegean_unet').is_dir() and (place / 'aegean_unet.zip').exists():
        with zipfile.ZipFile(place / 'aegean_unet.zip') as z:
            z.extractall(place)
    if (place / 'aegean_unet' / 'cube.npy').exists():
        IMAGE = place / 'aegean_unet'
        break
else:
    raise FileNotFoundError('no aegean_unet/cube.npy here or one folder up: upload aegean_unet.zip')
print('image:', IMAGE.resolve())

In [ ]:
# The bitstream in this folder: justounetsimple_opt.bit or justounetsimple_opt_zynq7020.bit.
bits = sorted(Path('.').glob('*.bit'))
if len(bits) != 1:
    raise FileNotFoundError(f'expected one .bit in this folder, found {[b.name for b in bits]}')
overlay = Overlay(str(bits[0]))     # programs the FPGA and sets the clock
# The kernel's registers: where this board's block design put them (0xA000_0000 on the
# ZCU104, 0x4000_0000 on the PYNQ-Z2), from the .hwh.
REGS = overlay.ip_dict['kernel']['phys_addr']
# DDR for the driver: contiguous, and Linux leaves it alone while this notebook holds it.
# As much as PYNQ can give, up to 128 MB (the PYNQ-Z2 has 512 MB of RAM in all); with less,
# the driver runs the image in strips of rows, with the same answer.
for mb in (128, 64, 32):
    try:
        ddr = allocate(shape=(mb * 1024 * 1024,), dtype=np.uint8)
        break
    except Exception as error:
        print(f'{mb} MB not available ({error})')
else:
    raise MemoryError('PYNQ has not even 32 MB free: shut down the other notebooks, or reboot')
print(f'{bits[0].name}: registers at {REGS:#x}; driver DDR: {mb} MB at {ddr.physical_address:#x}')

In [ ]:
command = ['./jopt_driver', '--devmem-regs', f'{REGS:#x}',
           '--devmem-memory', f'{ddr.physical_address:#x}:{ddr.nbytes:#x}',
           '--input', str(IMAGE / 'cube.npy'),
           '--labels', str(IMAGE / 'driver_labels.npy'),
           '--scores', str(IMAGE / 'driver_scores.npy')]
if (IMAGE / 'reference_fixed.npy').exists():
    command += ['--reference', str(IMAGE / 'reference_fixed.npy')]
if os.geteuid() != 0:                 # /dev/mem needs root
    command = ['sudo', '-n'] + command
run = subprocess.run(command, capture_output=True, text=True)
print(run.stdout + run.stderr)
if run.returncode == 2:
    raise RuntimeError('WRONG ANSWER: the FPGA differs from reference_fixed.npy (see above)')
if run.returncode != 0:
    raise RuntimeError('the driver failed (see above, and "If it failed" at the end)')
print('OK')

In [ ]:
# The driver's classes next to the PC's (reference_fixed.npy, lane 3): 0 cloud, 1 land, 2 sea.
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180]], dtype=np.uint8)
panels = {'driver (FPGA)': COLORS[np.load(IMAGE / 'driver_labels.npy')]}
if (IMAGE / 'reference_fixed.npy').exists():
    panels['PC integer model'] = COLORS[np.load(IMAGE / 'reference_fixed.npy')[..., 3]]
fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5), squeeze=False)
for ax, (title, img) in zip(axes[0], panels.items()):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.show()     # cloud white, land green, sea blue

**If it failed**
- `cannot open /dev/mem`: Jupyter is not root and `sudo -n` needs a password. Run the
  command printed above in a terminal with `sudo`.
- `cannot mmap /dev/mem ... Operation not permitted`: this Linux does not allow `/dev/mem`
  on ordinary RAM. See "ZCU104 with PYNQ" in `software/driver/README.md` (`mem=1536M`).
- `the kernel is not idle`: run the `Overlay` cell again.
- A timeout: the bitstream and `xjustounetsimple_opt_hw.h` here must be from the same build;
  re-make the package with `software/driver/scripts/package.sh zcu104` (or `zynq7020`).
- `FATAL: kernel too old`: this board's Linux is older than the one the driver was built
  for; build it with Ubuntu's cross compiler instead of Vitis's (README.md).